## 第 7 课：Block Copy 与 SMEM

> 对应原文：笔记 (6)。引入 SMEM 内存层级，建立 GMEM → SMEM → RMEM 的二级拷贝流水，解决 GMEM 冗余访存问题，并用 NCU 分析 SMEM 的 Bank Conflict。
> 环境：CUTLASS 4.3.4，SM90。

## 学习目标

- 理解 SMEM 硬件特性：32 个 Bank、transaction、wavefront、**Bank Conflict**、Broadcast/Multicast；
- 会用 `make_tiled_copy` 构建 GMEM→SMEM（`cp.async`）和 SMEM→RMEM 两级 TiledCopy；
- 掌握设置 **ThrLayout / ValLayout** 的三个考虑因素；
- 会分配动态 SMEM 空间（`extern __shared__` + `cudaFuncSetAttribute`）；
- 会读 NCU 的 SMEM 指标，并手工核算 Bank Conflicts。

## 1. NV GPU 的共享内存访存特性

SMEM 位于每个 SM 内部，与 L1 共用物理存储。为提供高带宽，物理上划分为 **32 个等宽 Bank**（通常每 Bank 4 bytes）。延迟远低于 GMEM、带宽高得多。

- SM80 单 block 最大 SMEM 163 KB；SM90/SM100 227 KB。

![GPU 内存层级](assets/figs/fig_03_GPU_内存层级_图源自_FA_论文_.png)

**Bank Conflict**：同一 warp 的多个线程在**一次 transaction** 中访问**同一个 Bank 的不同地址**时发生，无法在一个 wavefront 中并行执行，被序列化为多个 wavefronts，降低有效带宽。

概念回顾：

- **transaction**：warp 一次访存请求按 128 bytes 粒度合并的单元（SMEM 不要求连续/对齐）；
- **wavefront**：L1/TEX 一次并行访存处理，一个时钟周期完成；理想情况一个 transaction = 一个 wavefront。

三个注意点：

1. Bank Conflict 从 **transaction 粒度**判断；
2. 凡是 SMEM 访存（SMEM↔GMEM、SMEM↔RMEM）都可能触发；
3. **Broadcast**：所有线程访问同一 Bank 的同一地址 → 一个 wavefront 无冲突；**Multicast**：多个线程访问同一 Bank 同一地址 → 也一个 wavefront。

CUTLASS 通过 **Swizzling**（内存布局变换）解决 Bank Conflict（下篇笔记，本篇只做拷贝）。

## 2. 二级 Tiling 和二级拷贝

引入 SMEM 后，数据通路变成两级：

In [ ]:
GMEM → SMEM（第一级拷贝，cp.async）→ RMEM（第二级拷贝）→ MMA → SMEM → GMEM

每一级拷贝的数据规模只取决于 TiledCopy 参数，**不一定等于同层 MMA 规模**：可以分多次拷贝、更细粒度手动控制 copy 循环。但大多数场景让同层 Copy 与 MMA 规模相同，易于理解。

![二级 Tiling 与二级拷贝的对应关系](assets/figs/fig_04_Global_Block_Tile_的二级_Tiling_以及_GMEM_SME.png)

## 3. Block Copy 实现

规格同笔记 (5)：问题规模 (128,128,64)，Block (128,128,64)，TiledMMA (32,32,32)，MMA atom (16,8,16)，256 线程。

### 3.1 GMEM → SMEM

`cp.async`：SM80 新增的 GMEM→SMEM 异步拷贝指令，直接从 GMEM 经 L2 到 SMEM，**不在 RMEM 中转**，通常是最优选择。单指令支持 128 bits 向量化：

In [ ]:
using Copy_G2S_op = SM80_CP_ASYNC_CACHEGLOBAL<cute::uint128_t>;
using CopyA_G2S_atom = Copy_Atom<Copy_G2S_op, ComputeTypeA>;

![cp.async 拷贝原理](assets/figs/fig_06_cp_async_拷贝原理_图源自_NVIDIA_Ampere_白皮书_.png)

**设置 ThrLayout / ValLayout 的三个因素**：

1. **数据规模**：Copy Tile 需能被数据规模整除且尽可能小，避免最后一个 Tile 越界（越界会报 IMA，TiledCopy 不帮你处理）；
2. **Copy Atom 的要求**：`cp.async` 要求 128 bits 内存连续。行连续矩阵 → ValLayout 按行拷 `(1,8)`；列连续 → `(8,1)`。2 bytes 元素时单线程元素数必须是 8 的倍数，否则编译期 static_assert 报错：
   ```text
   "TiledCopy uses too few vals for selected CopyAtom"
   ```
3. **访存连续性**：尽量让 Tile 数据在 128 bytes（cache line）粒度连续。例如 (128,128) 数据、Copy Tile (128,32) 时每行 64 bytes < 128，有一倍冗余访存；设成 (128,64) 每行 128 bytes 最优化。

本篇 gA 形状 (128,64)、行连续，ValLayout=(1,8)，ThrLayout 第二维 = min(64, kBlockK)/8：

In [ ]:
static constexpr int kThreadNum = size(TiledMMA{});
static constexpr int kBlockK_Copy = cute::min(64, kBlockK) / 8;

using TiledCopyA_G2S =
    decltype(make_tiled_copy(CopyA_G2S_atom{},
                             make_layout(make_shape(Int<kThreadNum / kBlockK_Copy>{}, Int<kBlockK_Copy>{}),
                                         make_stride(Int<kBlockK_Copy>{}, Int<1>{})),
                             make_layout(make_shape(Int<1>{}, Int<8>{}))));

### 3.2 SMEM → RMEM

规模就是 TiledMMA 的规模，构建流程同之前的 GMEM→RMEM（仍用 AutoVectorizingCopy）：

In [ ]:
using Copy_S2R_op = AutoVectorizingCopy;
using CopyA_S2R_atom = Copy_Atom<Copy_S2R_op, ComputeTypeA>;
using TiledCopyA_S2R = decltype(make_tiled_copy_A(CopyA_S2R_atom{}, TiledMMA{}));

### 3.3 结果写回（RMEM → SMEM → GMEM）

流程相同、方向相反：

In [ ]:
using CopyO_R2S_atom = Copy_Atom<Copy_R2S_op, OutType>;
using CopyO_S2G_atom = Copy_Atom<Copy_S2G_op, OutType>;
using TiledCopyO_R2S = decltype(make_tiled_copy_C(CopyO_R2S_atom{}, TiledMMA{}));
using TiledCopyO_S2G = decltype(make_tiled_copy(CopyO_S2G_atom{}, ...));  // 同 G2S 的三因素设置

### 3.4 创建 SMEM 空间

编译期计算布局与大小，输出 O 复用 A/B/C 的空间：

In [ ]:
using SmemLayoutA = decltype(make_layout(make_shape(Int<kTileM>{}, Int<kTileK>{}),
                                         make_stride(Int<kTileK>{}, Int<1>{})));
...
static constexpr int kShmSizeA = cosize(SmemLayoutA{}) * sizeof(ComputeTypeA);
...
static constexpr int kShmSize = cute::max(kShmSizeA + kShmSizeB + kShmSizeC, kShmSizeO);

launch 时动态指定（>48KB 需要 cudaFuncSetAttribute）：

In [ ]:
if (shm_size >= 48 * 1024) {
    cudaFuncSetAttribute(block_copy<...>, cudaFuncAttributeMaxDynamicSharedMemorySize, shm_size);
}
block_copy<...><<<grid, block, shm_size, stream>>>(...);

kernel 内创建 SMEM Tensor：

In [ ]:
extern __shared__ __align__(1024) uint8_t smem[];
uint8_t *Aptr_smem = smem;
uint8_t *Bptr_smem = smem + kShmSizeA;
uint8_t *Cptr_smem = smem + kShmSizeA + kShmSizeB;
uint8_t *Optr_smem = smem;

Tensor sA = make_tensor(make_smem_ptr((ComputeTypeA*)Aptr_smem), SmemLayoutA{});  // (kBlockM, kBlockK)

### 3.5 Kernel 代码骨架

In [ ]:
// 第一级：GMEM -> SMEM
Tensor tAgA_g2s = g2s_thr_copy_a.partition_S(gA);  // (CPY, CPY_M, CPY_K)
Tensor tAsA_g2s = g2s_thr_copy_a.partition_D(sA);
copy(g2s_tiled_copy_a, tAgA_g2s, tAsA_g2s);
// (需要 cp.async 同步 / fence，见后续笔记的 Pipelining)

// 第二级：SMEM -> RMEM
Tensor tAgA_s2r = s2r_thr_copy_a.partition_S(sA);
Tensor tArA_s2r = s2r_thr_copy_a.partition_D(tCrA);
copy(s2r_tiled_copy_a, tAgA_s2r, tArA_s2r);

gemm(tiled_mma, tCrC, tCrA, tCrB, tCrC);

// 写回：RMEM -> SMEM -> GMEM

## 4. NCU 算子分析

### 4.1 GMEM → SMEM：问题已解决

`cp.async` 对应 SASS 的 `LDGSTS`。256 线程、A/B 各 (128,64)，每线程 32+32=64 个元素 → 8 条 cp.async。ncu 不再报访存问题：一条指令合并为 4 个 transactions，1 个 transaction 恰好写入 32 个 bank，无 Bank Conflict——**第一级拷贝达到最优**。

![ncu 展示的 8 条 LDGSTS 指令](assets/figs/fig_07_ncu_展示的_8_条_LDGSTS_指令.png)

### 4.2 SMEM → RMEM：引入新问题

第二级拷贝受 MMA 数据排布限制，字长只能是 32 bits → `ld.shared.u32` / `LDS`。每个 transaction 访问了同一 Bank 的 8 个数据 → 触发 8 个 wavefronts，其中 7/8 是多余的——**Bank Conflict**。8 个 warp 各 1 个 transaction，理想 8 个 wavefronts，实际 64 个。

![SMEM -> RMEM 的读取内存排布](assets/figs/fig_09_SMEM____RMEM_过程中_SMEM_的读取内存排布.png)

![ncu 会报告 SMEM 的访存问题，例如 Bank Conflict](assets/figs/fig_10_ncu_会报告_SMEM_的访存问题_例如_Bank_Conflict.png)

### 4.3 SMEM 指标手工核算

![ncu 展示的 SMEM 访存指标统计](assets/figs/fig_11_ncu_展示的_SMEM_访存指标统计.png)

1. **Shared Load（LDS）**：SMEM→RMEM 需 (128×64)/(32×32)=8 个 A Tile + 8 个 B Tile，每 warp 每 Tile 8+4 条 LDS → 每 warp 96 条、8 warp 共 **768 条**；SMEM→GMEM 用 LDS.128，64 条。总 832 条。
2. **Shared Store（STS）**：RMEM→SMEM 共 256 条，每条 1 transaction、8 wavefronts。
3. **Shared Store From Global Load（LDGSTS）**：64 条，每条 4 transactions、4 wavefronts，无 Bank Conflict。

总 wavefronts：`768×8 + 64×4 = 6400`，其中 Bank Conflict 多产生 `768×7 = 5376` 个——与 ncu 表格吻合。

**结论**：引入 SMEM 解决了 GMEM 冗余访存，但又引入 SMEM Bank Conflict。下一步用 **Swizzling** 解决（下一篇）。

## 同时回答

1. Bank Conflict 是什么？transaction 和 wavefront 是什么关系？Broadcast / Multicast 各解决什么？
2. 设置 ThrLayout / ValLayout 要考虑哪三个因素？为什么 Copy Tile 需要能被数据规模整除？
3. 手工核算：本篇中 Shared Load 的 768 条 LDS 和 5376 个多余 wavefronts 是怎么算出来的？

把代码和三个答案发给我，我继续审查。